# NetCDF Three-Way Comparison

Side-by-side comparison of three datasets:

| Column | Dataset | Description |
|--------|---------|-------------|
| **A** | `diffusion-interpolation/_saved/outputs/` | Diffusion interpolation (11 members) |
| **B** | `metno_interpolator/data/interpolator_MEPS_2024-01_11_memb.nc` | MetNo interpolator (11 members) |
| **C** | `metno_interpolator/data/MEPS_2024-01.nc` | Single-member reference |

**Note:** Dataset C has 264 time steps (vs 241 in A/B), `ensemble=1`, and the wind-speed variable is named `ws10m` (vs `ws10` in A/B). The notebook handles all of this automatically.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# ── File paths ──────────────────────────────────────────────────────
FILE_A = "/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_saved/outputs/interpolator_MEPS_2024-01_11_memb.nc"
FILE_A = "/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_saved/outputs_answer/interpolator_MEPS_2024-01_11_memb.nc"
FILE_A = "/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_saved/output-bis-2nd-report-00/interpolator_MEPS_2024-01_11_memb.nc"
FILE_B = "/project/home/p200177/DE_371/metno_interpolator/data/interpolator_MEPS_2024-01_11_memb.nc"
FILE_C = "/project/home/p200177/DE_371/metno_interpolator/data/MEPS_2024-01.nc"

LABEL_A = "Diffusion Interp."
LABEL_B = "MetNo Interp."
LABEL_C = "Reference (single)"

ds_a = xr.open_dataset(FILE_A)
ds_b = xr.open_dataset(FILE_B)
ds_c = xr.open_dataset(FILE_C)

print("── Dataset A ──")
print(f"  Variables : {list(ds_a.data_vars)}")
print(f"  Dims      : {dict(ds_a.sizes)}")
print("── Dataset B ──")
print(f"  Variables : {list(ds_b.data_vars)}")
print(f"  Dims      : {dict(ds_b.sizes)}")
print("── Dataset C (reference) ──")
print(f"  Variables : {list(ds_c.data_vars)}")
print(f"  Dims      : {dict(ds_c.sizes)}")

## Align Time Axes

Dataset C has 264 time steps (with `units = seconds since 1970-01-01`), while A and B have 241. We align on matching time values so comparisons are apples-to-apples.

In [ ]:
# Convert C's time to the same representation used by A/B
# A/B store int64 time values; C stores doubles with a units attribute.
# We decode C's time to numpy datetime64 for matching.

def decode_time(ds):
    """Return time coordinate as numpy datetime64 array."""
    t = ds["time"]
    if np.issubdtype(t.dtype, np.datetime64):
        return t.values
    # If xarray auto-decoded, it's already datetime64
    # Otherwise try pd.to_datetime on raw values
    try:
        return pd.to_datetime(t.values).values
    except Exception:
        return t.values  # fallback: use raw values

time_a = decode_time(ds_a)
time_b = decode_time(ds_b)
time_c = decode_time(ds_c)

# Find common times between A/B and C
common_times_ac = np.intersect1d(time_a, time_c)
print(f"Time steps — A: {len(time_a)}, B: {len(time_b)}, C: {len(time_c)}")
print(f"Common times (A∩C): {len(common_times_ac)}")

if len(common_times_ac) == 0:
    print("\n⚠ No overlapping decoded times found.")
    print("  Falling back to index-based alignment (first 241 steps).")
    ALIGN_BY_INDEX = True
else:
    ALIGN_BY_INDEX = False

In [ ]:
def get_time_index_in_c(time_idx_ab):
    """Map a time index from A/B to the corresponding index in C."""
    if ALIGN_BY_INDEX:
        # Simple: same index if within bounds
        if time_idx_ab < len(time_c):
            return time_idx_ab
        return None
    else:
        target = time_a[time_idx_ab]
        matches = np.where(time_c == target)[0]
        return int(matches[0]) if len(matches) > 0 else None

## Configuration

Edit the cell below to choose which variables, time indices, and ensemble member to compare.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  USER CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

# Variables to compare.
# Use the A/B names; the mapping to C's names is handled below.
VARIABLES = ["10u", "10v", "2t", "msl", "tp", "ws10"]

# Time indices (in A/B indexing, 0–240)
TIME_INDICES = [3, 63, 123, 183, 183+48]

# Ensemble member for A and B (0–10). C only has member 0.
ENSEMBLE_IDX = 1

# Use lat/lon for spatial axes
USE_LATLON = True

# ── Variable-name mapping: A/B name → C name ──────────────────────
# Dataset C uses 'ws10m' instead of 'ws10'; everything else matches.
VAR_MAP_C = {
    "10u": "10u",
    "10v": "10v",
    "2t": "2t",
    "msl": "msl",
    "tp": "tp",
    "ws10": "ws10m",
}

# Colormaps per variable
CMAPS = {
    "10u": "RdBu_r",
    "10v": "RdBu_r",
    "2t": "RdYlBu_r",
    "msl": "viridis",
    "tp": "YlGnBu",
    "ws10": "magma",
}

# Pretty labels
VAR_LABELS = {
    "10u": "10m U-wind [m/s]",
    "10v": "10m V-wind [m/s]",
    "2t": "2m Temperature [K]",
    "msl": "Mean Sea-Level Pressure [Pa]",
    "tp": "Total Precipitation",
    "ws10": "10m Wind Speed [m/s]",
}

## Spatial Comparison: A | B | Reference C | Diff (A−C) | Diff (B−C)

In [ ]:
def get_field(ds, var, time_idx, ens_idx):
    """Extract a 2-D (y, x) field."""
    return ds[var].isel(time=time_idx, ensemble=ens_idx).values


def plot_comparison_3way(var, time_idx, ens_idx=ENSEMBLE_IDX):
    """
    5-panel row:  A  |  B  |  C (ref)  |  A−C  |  B−C
    """
    var_c = VAR_MAP_C.get(var, var)
    time_idx_c = get_time_index_in_c(time_idx)

    field_a = get_field(ds_a, var, time_idx, ens_idx)
    field_b = get_field(ds_b, var, time_idx, ens_idx)

    has_ref = (time_idx_c is not None) and (var_c in ds_c)
    if has_ref:
        field_c = get_field(ds_c, var_c, time_idx_c, 0)  # C has only ensemble=0
        diff_ac = field_a - field_c
        diff_bc = field_b - field_c
        ncols = 5
    else:
        ncols = 3  # fallback: just A, B, A−B

    # Shared color range for the field panels
    fields = [field_a, field_b] + ([field_c] if has_ref else [])
    vmin = np.nanmin(fields)
    vmax = np.nanmax(fields)

    cmap = CMAPS.get(var, "RdBu_r")
    label = VAR_LABELS.get(var, var)
    time_val = ds_a["time"].values[time_idx]

    lat = ds_a["lat"].values if (USE_LATLON and "lat" in ds_a) else None
    lon = ds_a["lon"].values if (USE_LATLON and "lon" in ds_a) else None

    fig, axes = plt.subplots(1, ncols, figsize=(5.2 * ncols, 5.5))

    def _plot(ax, field, title, vmin, vmax, cmap):
        if lat is not None:
            im = ax.pcolormesh(lon, lat, field, vmin=vmin, vmax=vmax,
                               cmap=cmap, shading="auto")
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
        else:
            im = ax.imshow(field, origin="lower", vmin=vmin, vmax=vmax,
                           cmap=cmap, aspect="equal")
            ax.set_xlabel("x")
            ax.set_ylabel("y")
        ax.set_title(title, fontsize=10)
        return im

    # ── Field panels ──
    im = _plot(axes[0], field_a, LABEL_A, vmin, vmax, cmap)
    _plot(axes[1], field_b, LABEL_B, vmin, vmax, cmap)

    if has_ref:
        _plot(axes[2], field_c, LABEL_C, vmin, vmax, cmap)
        fig.colorbar(im, ax=list(axes[:3]), label=label, shrink=0.85,
                     orientation="horizontal", pad=0.12)

        # ── Difference panels ──
        dmax = max(np.nanmax(np.abs(diff_ac)), np.nanmax(np.abs(diff_bc)), 1e-12)
        im_d1 = _plot(axes[3], diff_ac, f"A − Ref", -dmax, dmax, "RdBu_r")
        im_d2 = _plot(axes[4], diff_bc, f"B − Ref", -dmax, dmax, "RdBu_r")
        fig.colorbar(im_d2, ax=list(axes[3:]), label=f"Δ {label}", shrink=0.85,
                     orientation="horizontal", pad=0.12)
    else:
        # No reference available — show A−B instead
        diff_ab = field_a - field_b
        dmax = np.nanmax(np.abs(diff_ab))
        fig.colorbar(im, ax=list(axes[:2]), label=label, shrink=0.85,
                     orientation="horizontal", pad=0.12)
        im_d = _plot(axes[2], diff_ab, "A − B", -dmax, dmax, "RdBu_r")
        fig.colorbar(im_d, ax=axes[2], label=f"Δ {label}", shrink=0.85,
                     orientation="horizontal", pad=0.12)

    ref_note = f" (ref time idx {time_idx_c})" if has_ref else " [no ref]"
    fig.suptitle(
        f"{label}  ·  time idx {time_idx} ({time_val})  ·  ens {ens_idx}{ref_note}",
        fontsize=12, fontweight="bold", y=1.03,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Generate all comparison plots ──────────────────────────────────
for var in VARIABLES:
    for tidx in TIME_INDICES:
        plot_comparison_3way(var, tidx)

## Summary Statistics vs Reference

In [ ]:
rows = []
for var in VARIABLES:
    var_c = VAR_MAP_C.get(var, var)
    has_ref = var_c in ds_c

    a = ds_a[var].isel(ensemble=ENSEMBLE_IDX)
    b = ds_b[var].isel(ensemble=ENSEMBLE_IDX)

    if has_ref:
        c = ds_c[var_c].isel(ensemble=0)
        # Align on common time indices
        n = min(a.sizes["time"], c.sizes["time"])
        a_v = a.isel(time=slice(0, n)).values
        b_v = b.isel(time=slice(0, n)).values
        c_v = c.isel(time=slice(0, n)).values
        diff_ac = a_v - c_v
        diff_bc = b_v - c_v
        rows.append({
            "Variable": VAR_LABELS.get(var, var),
            "Mean Ref": f"{np.nanmean(c_v):.4f}",
            "Mean A": f"{np.nanmean(a_v):.4f}",
            "Mean B": f"{np.nanmean(b_v):.4f}",
            "RMSE(A−Ref)": f"{np.sqrt(np.nanmean(diff_ac**2)):.4e}",
            "RMSE(B−Ref)": f"{np.sqrt(np.nanmean(diff_bc**2)):.4e}",
            "MaxErr(A−Ref)": f"{np.nanmax(np.abs(diff_ac)):.4e}",
            "MaxErr(B−Ref)": f"{np.nanmax(np.abs(diff_bc)):.4e}",
        })
    else:
        a_v = a.values
        b_v = b.values
        diff_ab = a_v - b_v
        rows.append({
            "Variable": VAR_LABELS.get(var, var),
            "Mean Ref": "N/A",
            "Mean A": f"{np.nanmean(a_v):.4f}",
            "Mean B": f"{np.nanmean(b_v):.4f}",
            "RMSE(A−Ref)": "N/A",
            "RMSE(B−Ref)": "N/A",
            "MaxErr(A−Ref)": "N/A",
            "MaxErr(B−Ref)": "N/A",
        })

stats_df = pd.DataFrame(rows)
stats_df.style.set_caption("Error statistics against single-member reference")

## Domain-Mean Time Series (A, B, Reference)

In [ ]:
fig, axes = plt.subplots(len(VARIABLES), 1, figsize=(14, 3.5 * len(VARIABLES)),
                         sharex=False)
if len(VARIABLES) == 1:
    axes = [axes]

for ax, var in zip(axes, VARIABLES):
    var_c = VAR_MAP_C.get(var, var)

    mean_a = ds_a[var].isel(ensemble=ENSEMBLE_IDX).mean(dim=["y", "x"]).values
    mean_b = ds_b[var].isel(ensemble=ENSEMBLE_IDX).mean(dim=["y", "x"]).values

    ax.plot(np.arange(len(mean_a)), mean_a, label=LABEL_A, linewidth=1.5)
    ax.plot(np.arange(len(mean_b)), mean_b, label=LABEL_B, linewidth=1.5, linestyle="--")

    if var_c in ds_c:
        mean_c = ds_c[var_c].isel(ensemble=0).mean(dim=["y", "x"]).values
        ax.plot(np.arange(len(mean_c)), mean_c, label=LABEL_C,
                linewidth=2, linestyle=":", color="black", alpha=0.7)

    ax.set_ylabel(VAR_LABELS.get(var, var), fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time index")
fig.suptitle(f"Domain-Mean Time Series  ·  Ensemble member {ENSEMBLE_IDX}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Ensemble Spread vs Reference

For A and B (11 members), show the ensemble mean ± 1σ against the single-member reference.

In [ ]:
fig, axes = plt.subplots(len(VARIABLES), 2, figsize=(16, 3.5 * len(VARIABLES)),
                         sharex=True)
if len(VARIABLES) == 1:
    axes = axes.reshape(1, -1)

for row, var in enumerate(VARIABLES):
    var_c = VAR_MAP_C.get(var, var)

    for col, (ds, lbl) in enumerate([(ds_a, LABEL_A), (ds_b, LABEL_B)]):
        ax = axes[row, col]
        # Spatial mean per ensemble member → (time, ensemble)
        spatial_mean = ds[var].mean(dim=["y", "x"])  # (time, ensemble)
        ens_mean = spatial_mean.mean(dim="ensemble").values
        ens_std = spatial_mean.std(dim="ensemble").values
        t = np.arange(len(ens_mean))

        ax.plot(t, ens_mean, label=f"{lbl} mean", linewidth=1.5)
        ax.fill_between(t, ens_mean - ens_std, ens_mean + ens_std,
                        alpha=0.25, label="±1σ")

        if var_c in ds_c:
            mean_c = ds_c[var_c].isel(ensemble=0).mean(dim=["y", "x"]).values
            ax.plot(np.arange(len(mean_c)), mean_c, label=LABEL_C,
                    linewidth=1.5, linestyle=":", color="black", alpha=0.7)

        ax.set_title(f"{lbl} — {VAR_LABELS.get(var, var)}", fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

axes[-1, 0].set_xlabel("Time index")
axes[-1, 1].set_xlabel("Time index")
fig.suptitle("Ensemble Spread (mean ± 1σ) vs Single-Member Reference",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Interactive Explorer

In [ ]:
try:
    var_dropdown = widgets.Dropdown(options=VARIABLES, value=VARIABLES[0],
                                    description="Variable:")
    time_slider = widgets.IntSlider(value=0, min=0, max=ds_a.sizes["time"] - 1,
                                    step=1, description="Time idx:")
    ens_slider = widgets.IntSlider(value=0, min=0, max=ds_a.sizes["ensemble"] - 1,
                                   step=1, description="Ensemble:")

    ui = widgets.VBox([var_dropdown, time_slider, ens_slider])
    out = widgets.interactive_output(
        plot_comparison_3way,
        {"var": var_dropdown, "time_idx": time_slider, "ens_idx": ens_slider},
    )
    display(ui, out)
except Exception as e:
    print(f"Interactive widgets not available ({e}). Use the static plots above.")